In [1]:
import os
import pandas as pd
import numpy as np

### Segmentation

In [41]:
import os
import pandas as pd

Dir_github = r'X:\PROJECTS\ABC@UofSC\3-Experiments\NSF_valence\github\nsf_valence'
Dir_segmentations = os.path.join(Dir_github, 'derivatives', 'behavioral', 'segmentation')

Dir_ratings = os.path.join(Dir_github,'derivatives','behavioral','ratings','audio')
df_ratings_audio = pd.read_csv(os.path.join(Dir_ratings,'ratings_audio_att2_comp2.csv'), dtype={'idx': str,'ethnicity':str},keep_default_na=False)

List_files = [x for x in os.listdir(Dir_segmentations) if x.startswith('segmentation_')]
df_segmentations = pd.concat(
    [pd.read_csv(os.path.join(Dir_segmentations, file)) for file in List_files],
    ignore_index=True
)

Dir_timestamps = os.path.join(Dir_github, 'derivatives', 'behavioral', 'timestamps_segment')
List_files = [x for x in os.listdir(Dir_timestamps) if x.startswith('timestamps_')]
df_timestamps = pd.concat(
    [pd.read_csv(os.path.join(Dir_timestamps, file)) for file in List_files],
    ignore_index=True
)

df_timestamps['duration'] = df_timestamps['offset'] - df_timestamps['onset']

# Build summary
df_summary = df_segmentations.groupby('narrative').agg(
    Total_sentences=('sentID', 'max'),
    Total_segments=('segID', 'max'),
    Total_words=('Nwords', 'sum')
)

df_summary['Words/segment'] = df_summary['Total_words'] / df_summary['Total_segments']
df_summary['Seconds/segment'] = df_timestamps.groupby('narrative')['duration'].mean()

# SDs for each narrative
sd_words = df_segmentations.groupby('narrative')['Nwords'].std()
sd_seconds = df_timestamps.groupby('narrative')['duration'].std()

# Create formatted display table
df_display = df_summary.copy()

df_display['Words/segment'] = [
    f"{df_summary.loc[i, 'Words/segment']:.2f} ± {sd_words.loc[i]:.2f}"
    for i in df_display.index
]
df_display['Seconds/segment'] = [
    f"{df_summary.loc[i, 'Seconds/segment']:.2f} ± {sd_seconds.loc[i]:.2f}"
    for i in df_display.index
]

# Add average row with SD across narratives
avg_row = pd.Series(
    {
        'Total_sentences': f"{df_summary['Total_sentences'].mean():.2f} ± {df_summary['Total_sentences'].std():.2f}",
        'Total_segments': f"{df_summary['Total_segments'].mean():.2f} ± {df_summary['Total_segments'].std():.2f}",
        'Total_words': f"{df_summary['Total_words'].mean():.2f} ± {df_summary['Total_words'].std():.2f}",
        'Words/segment': f"{df_summary['Words/segment'].mean():.2f} ± {df_summary['Words/segment'].std():.2f}",
        'Seconds/segment': f"{df_summary['Seconds/segment'].mean():.2f} ± {df_summary['Seconds/segment'].std():.2f}",
    },
    name='Average'
)

df_display = pd.concat([df_display, avg_row.to_frame().T])
df_display = df_display.reset_index()

df_voices = pd.DataFrame({'audio':df_ratings_audio['audio'].unique()})
df_voices[['voice','narrative']] = df_voices['audio'].str.split('_',expand=True)
df_voices.loc[df_voices['voice'].isin(['Jacob','Jack', 'WavenetB']),'voice_sex'] = 'M'
df_voices.loc[df_voices['voice'].isin(['Brandi', 'Allison', 'WavenetF','Jewell']),'voice_sex'] = 'F'
df_voices = pd.merge(left = df_display,left_on = 'index',
                      right = df_voices[['narrative','voice','voice_sex']],right_on = 'narrative',how='left')

df_voice_summary = (
    df_voices
    .sort_values(['narrative', 'voice_sex'], ascending=[True, False])  # M first (since M > F)
    .groupby('narrative')
    .apply(lambda x: ' / '.join(
        x.apply(lambda r: f"{r['voice']} ({r['voice_sex']})", axis=1)
    ))
    .reset_index(name='Voice')
)
df_display = df_display.merge(
    df_voice_summary,
    left_on='index',
    right_on='narrative',
    how='left'
).drop(columns='narrative')[['index','Voice', 'Total_sentences', 'Total_segments', 'Total_words',
       'Words/segment', 'Seconds/segment']]

Dir_output = os.path.join(Dir_github,'derivatives','behavioral','figures&tables') 
os.makedirs(Dir_output,exist_ok=True) 
df_display.to_csv(os.path.join(Dir_output,'Table1_segmentation_summary.csv'),encoding="utf-8-sig",) 


C:\Users\XuanR\AppData\Local\Temp\ipykernel_45980\1647852487.py:77: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: ' / '.join(


In [36]:
df_display.columns

Index(['index', 'Total_sentences', 'Total_segments', 'Total_words',
       'Words/segment', 'Seconds/segment'],
      dtype='object')

C:\Users\XuanR\AppData\Local\Temp\ipykernel_45980\3081497377.py:5: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: ' / '.join(


,index,Voice,Total_sentences,Total_segments,Total_words,Words/segment,Seconds/segment
0,AgedMother,Jacob (M) / Allison (F),51,128,895,6.99 ± 3.19,2.29 ± 1.14
1,Aqua,Jacob (M) / Brandi (F),36,147,990,6.73 ± 2.99,2.02 ± 0.95
2,BiggestWastes,Jack (M) / Allison (F),58,152,920,6.05 ± 2.74,1.84 ± 0.93
3,BradfordBoar,Jacob (M) / Brandi (F),54,157,1073,6.83 ± 3.25,1.93 ± 0.94
4,BulliesPronouns,WavenetB (M) / Allison (F),48,158,999,6.32 ± 2.97,1.97 ± 0.98
5,BulliesRepeatedNames,WavenetB (M) / Allison (F),48,158,1001,6.34 ± 2.96,1.98 ± 0.95
6,Elvis,Jack (M) / Brandi (F),44,157,1013,6.45 ± 3.06,1.70 ± 0.96
7,EmotionSentences,Jack (M) / Brandi (F),59,85,362,4.26 ± 1.57,1.23 ± 0.50
8,EmpireState,Jacob (M) / Brandi (F),49,113,962,8.51 ± 4.30,2.88 ± 1.73
9,FairyTree,Jacob (M) / Brandi (F),64,110,805,7.32 ± 3.44,2.23 ± 1.11


### Samples

In [3]:
Dir_ratings = os.path.join(Dir_github,'derivatives','behavioral','ratings','audio')
df_ratings_audio = pd.read_csv(os.path.join(Dir_ratings,'ratings_audio_att2_comp2.csv'), dtype={'idx': str,'ethnicity':str},keep_default_na=False)

df = df_ratings_audio.loc[df_ratings_audio['segID'].eq(1)].copy()
df[['voice', 'narrative']] = df['audio'].str.split('_', n=1, expand=True)
# df.rename({'V':'Valence','A':'Arousal','age':'Age','sex':},axis=1,inplace=True)
group_order = ['young', 'middle', 'old']
audio_index = (
    df[['narrative', 'voice']]
    .drop_duplicates()
    .sort_values(['narrative', 'voice'])
    .set_index(['narrative', 'voice'])
    .index
)
def mean_sd(s):
    s = pd.to_numeric(s, errors='coerce').dropna()
    if len(s) == 0:
        return np.nan
    if len(s) == 1:
        return f"{s.iloc[0]:.2f} ± NA"
    return f"{s.mean():.2f} ± {s.std(ddof=1):.2f}"

def sex_counts(s):
    x = s.astype(str).str.strip().str.lower()
    male = x.isin(['male', 'm']).sum()
    female = x.isin(['female', 'f']).sum()
    return f"{male} / {female}"

parts = []

for g in group_order:
    sub = (
        df.loc[df['group'].eq(g)]
          .groupby(['narrative', 'voice'], sort=False)
          .agg(
              N=('audio', 'size'),
              Age=('age', mean_sd),
              Sex=('sex', sex_counts),
            #   Valence=('V', mean_sd),
            #   Arousal=('A', mean_sd)
          )
          .reindex(audio_index)
    )
    
    sub.columns = pd.MultiIndex.from_product([[g], sub.columns])
    parts.append(sub)

summary_table = pd.concat(parts, axis=1)
summary_table.index.names = ['narrative', 'voice']


Dir_output = os.path.join(Dir_github,'derivatives','behavioral','figures&tables') 
# summary_table.to_csv(os.path.join(Dir_output,'Table2_sample_audio.csv'),encoding="utf-8-sig",) 
# summary_table.to_excel(os.path.join(Dir_output,'Table2_sample_audio.xlsx'), index=True)
summary_table

young                       middle  \
                                    N           Age     Sex      N   
narrative              voice                                         
AgedMother             Allison     20  22.00 ± 4.95  8 / 12     16   
                       Jacob       19  21.74 ± 4.75  3 / 16     15   
Aqua                   Brandi      19  24.37 ± 6.04  5 / 14     17   
                       Jacob       18  23.67 ± 5.91  5 / 13     15   
BiggestWastes          Allison     21  22.48 ± 6.10  7 / 14     19   
                       Jack        15  21.87 ± 5.50  5 / 10     15   
BradfordBoar           Brandi      17  20.24 ± 2.44  3 / 14     18   
                       Jacob       16  21.81 ± 4.18  3 / 13     15   
BulliesPronouns        Allison     19  23.79 ± 5.60  6 / 13     17   
                       WavenetB    16  23.50 ± 6.19  3 / 13     16   
BulliesRepeatedNames   Allison     17  20.41 ± 1.94  4 / 13     15   
                       WavenetB    16  22.69 ± 5.80  4 / 12     15   
Elvis                  Brandi      17  22.59 ± 4.53  7 / 10     16   
                       Jack        16  25.62 ± 7.17  4 / 12     16   
EmotionSentences       Brandi      20  19.35 ± 1.04  2 / 18     16   
                       Jack        19  20.74 ± 2.18  5 / 14     17   
EmpireState            Brandi      17  22.24 ± 4.79  4 / 13     14   
                       Jacob       18  21.89 ± 5.23  6 / 12     15   
FairyTree              Brandi      19  21.42 ± 4.75  9 / 10     15   
                       Jacob       19  24.42 ± 6.66  8 / 11     15   
FireSelected           Jacob       16  23.75 ± 6.21  3 / 13     16   
                       WavenetF    18  26.33 ± 6.66  5 / 13     15   
FirstConcert           Brandi      19  23.37 ± 6.16  7 / 12     15   
                       Jack        18  25.17 ± 6.19  3 / 15     16   
HappyDay               Brandi      18  22.94 ± 6.61  5 / 13     16   
                       Jacob       16  24.06 ± 7.47  4 / 12     16   
HowCouldYou            Brandi      17  25.71 ± 7.49  5 / 12     15   
                       Jacob       19  24.05 ± 6.11  4 / 15     15   
IronyParas             WavenetB    19  24.05 ± 6.99  5 / 14     16   
                       WavenetF    20  25.10 ± 6.69  6 / 14     16   
Koko                   Jacob       17  24.76 ± 5.76  6 / 11     17   
                       WavenetF    21  26.90 ± 6.39  6 / 15     15   
MetaphorsPronouns      Allison     16  23.50 ± 5.15  4 / 12     16   
                       Jack        17  25.76 ± 7.22  4 / 13     14   
MetaphorsRepeatedNames Allison     16  26.81 ± 7.42  10 / 6     16   
                       Jack        15  24.87 ± 6.73  3 / 12     18   
ProfessionalDecline    Jack        15  22.87 ± 6.00  3 / 12     15   
                       Jewell      14  22.64 ± 6.11   5 / 9     14   
Roswell                Brandi      16  27.62 ± 7.64   7 / 9     17   
                       Jack        18  26.39 ± 7.85  2 / 16     15   
TaleofTwoCountries     Brandi      18  23.56 ± 5.83  6 / 12     15   
                       Jacob       16  23.25 ± 6.73  5 / 11     17   
TigerandLion           Brandi      17  27.59 ± 7.91  5 / 12     15   
                       Jacob       17  24.88 ± 6.55  5 / 12     15   
UserFriendly           Brandi      17  25.76 ± 7.42  6 / 11     14   
                       Jacob       16  22.31 ± 5.41   7 / 9     15   

                                                      old                \
                                          Age     Sex   N           Age   
narrative              voice                                              
AgedMother             Allison   47.00 ± 4.46   9 / 7  16  65.94 ± 3.89   
                       Jacob     51.20 ± 5.29  3 / 12  15  66.27 ± 3.45   
Aqua                   Brandi    47.24 ± 4.29  7 / 10  14  66.64 ± 4.63   
                       Jacob     50.33 ± 6.77   6 / 9  15  64.73 ± 4.20   
BiggestWastes          Allison   48.89 ± 5.23  11 / 8  14  65.43 ± 3.96   
 

In [119]:
df = df_ratings_audio.loc[df_ratings_audio['segID'].eq(1)].copy()

# Keep one row per participant
df_sub = df.drop_duplicates(subset='subID')

# def sex_counts(s):
#     x = s.astype(str).str.strip().str.lower()
#     male = x.isin(['male', 'm']).sum()
#     female = x.isin(['female', 'f']).sum()
#     return f"{male}/{female}"   # or f'="{male}/{female}"' for CSV

summary = (
    df_sub.groupby('group')
          .agg(
              N=('subID', 'nunique'),
              mean_age=('age', 'mean'),
              sd_age=('age', 'std'),
              sex=('sex', sex_counts)
          )
)

# Format age
summary['Age'] = summary.apply(lambda r: f"{r['mean_age']:.2f} ± {r['sd_age']:.2f}", axis=1)

# Final columns
summary = summary[['N', 'Age', 'sex']]

summary.columns = ['N', 'Age (mean ± SD)', 'Sex (M/F)']

# Optional ordering
summary = summary.loc[['young', 'middle', 'old']]

summary

,N,Age (mean ± SD),Sex (M/F)
group,,,
young,269,22.92 ± 5.67,73 / 196
middle,160,48.58 ± 5.37,73 / 87
old,100,65.71 ± 4.35,25 / 75


In [123]:
df_sub.groupby(['group','source']).count()

subID  idx  audio  audio_idx  segID    V    A   RT  age  sex  \
group  source                                                                   
middle Prolific    160  160    160        160    160  160  160  160  160  160   
old    Prolific    100  100    100        100    100  100  100  100  100  100   
young  Prolific     85   85     85         85     85   85   85   85   85   85   
       Sona        184  184    184        184    184  184  184  184  184  184   

                 age_prescreen  sex_prescreen  attention_correct  \
group  source                                                      
middle Prolific            160            160                160   
old    Prolific            100            100                100   
young  Prolific             85             85                 85   
       Sona                184            184                184   

                 attention_correct%  comp_correct  comp_correct%  ethnicity  \
group  source                                                                 
middle Prolific                 160           160            160        160   
old    Prolific                 100           100            100        100   
young  Prolific                  85            85             85         85   
       Sona                     184           184            184        184   

                 residence_states  
group  source                      
middle Prolific               160  
old    Prolific               100  
young  Prolific                85  
       Sona                   184

### sample reading

In [4]:
Dir_ratings = os.path.join(Dir_github,'derivatives','behavioral','ratings','text')
df_ratings_text = pd.read_csv(os.path.join(Dir_ratings,'ratings_text_att2_comp2.csv'), dtype={'idx': str,'ethnicity':str},keep_default_na=False)
df_ratings_text.loc[df_ratings_text['text']!='EmotionSentences'].to_csv(os.path.join(Dir_ratings,'ratings_text_att2_comp2.csv'),index=False)


df = df_ratings_text.loc[df_ratings_text['segID'].eq(1)].copy()
# df[['voice', 'narrative']] = df['audio'].str.split('_', n=1, expand=True)
# df.rename({'V':'Valence','A':'Arousal','age':'Age','sex':},axis=1,inplace=True)
group_order = ['young', 'middle', 'old']
audio_index = (
    df[['text']]
    .drop_duplicates()
    .sort_values(['text'])
    .set_index(['text'])
    .index
)


parts = []

for g in group_order:
    sub = (
        df.loc[df['group'].eq(g)]
          .groupby(['text'], sort=False)
          .agg(
              N=('text', 'size'),
              Age=('age', mean_sd),
              Sex=('sex', sex_counts),
            #   Valence=('V', mean_sd),
            #   Arousal=('A', mean_sd)
          )
          .reindex(audio_index)
    )
    
    sub.columns = pd.MultiIndex.from_product([[g], sub.columns])
    parts.append(sub)

summary_table = pd.concat(parts, axis=1)
summary_table.index.names = ['narrative']


Dir_output = os.path.join(Dir_github,'derivatives','behavioral','figures&tables') 
# summary_table.to_csv(os.path.join(Dir_output,'Table3_sample_text.csv'),encoding="utf-8-sig",) 
summary_table.to_excel(os.path.join(Dir_output,'Table3_sample_text.xlsx'), index=True)
summary_table

young                       middle                \
                           N           Age     Sex      N           Age   
narrative                                                                 
AgedMother                15  30.13 ± 6.42   9 / 6     24  48.12 ± 5.32   
Aqua                      22  25.41 ± 7.24  8 / 14     17  47.29 ± 4.38   
BiggestWastes             17  26.76 ± 8.42  4 / 13     15  50.80 ± 4.87   
BradfordBoar              16  26.06 ± 7.50  4 / 12     17  48.82 ± 5.93   
BulliesPronouns           18  29.00 ± 7.34  4 / 14     16  47.88 ± 4.81   
BulliesRepeatedNames      16  29.31 ± 5.68  5 / 11     16  48.88 ± 5.23   
Elvis                     18  28.50 ± 6.42  10 / 8     15  46.93 ± 5.91   
EmpireState               16  28.19 ± 7.39  6 / 10     15  49.80 ± 4.30   
FairyTree                 17  25.94 ± 6.80  7 / 10     16  51.31 ± 5.28   
FireSelected              20  25.75 ± 6.40  6 / 14     16  47.31 ± 6.76   
FirstConcert              15  29.33 ± 6.24   8 / 7     16  47.94 ± 5.41   
HappyDay                  16  25.69 ± 5.35   9 / 7     16  46.81 ± 5.55   
HowCouldYou               17  27.88 ± 7.65  6 / 11     16  50.88 ± 4.33   
IronyParas                17  29.41 ± 7.29  6 / 11     15  47.13 ± 5.01   
Koko                      15  32.07 ± 6.27   8 / 7     15  49.73 ± 5.78   
MetaphorsPronouns         19  27.05 ± 6.49  7 / 12     15  48.93 ± 4.89   
MetaphorsRepeatedNames    18  29.33 ± 7.80  6 / 12     15  48.80 ± 5.94   
ProfessionalDecline       17  29.41 ± 6.10  7 / 10     15  48.80 ± 5.16   
Roswell                   15  29.13 ± 4.27  5 / 10     16  47.81 ± 4.29   
TaleofTwoCountries        17  28.82 ± 7.34  7 / 10     16  45.81 ± 4.25   
TigerandLion              16  28.31 ± 7.87  4 / 12     15  46.93 ± 5.20   
UserFriendly              16  29.31 ± 6.15   8 / 8     15  48.60 ± 6.51   

                                old                        
                            Sex   N           Age     Sex  
narrative                                                  
AgedMother              10 / 14  17  65.47 ± 4.68  5 / 12  
Aqua                     10 / 7  21  65.24 ± 3.91  4 / 17  
BiggestWastes             6 / 9  16  64.75 ± 3.91  4 / 12  
BradfordBoar             10 / 7  15  64.93 ± 3.83   6 / 9  
BulliesPronouns          6 / 10  16  65.88 ± 4.22  3 / 13  
BulliesRepeatedNames     4 / 12  15  64.87 ± 3.11  4 / 11  
Elvis                     8 / 7  16  65.94 ± 3.87   7 / 9  
EmpireState               7 / 8  16  65.00 ± 3.14  5 / 11  
FairyTree                2 / 14  17  63.94 ± 2.49  3 / 14  
FireSelected              7 / 9  15  64.27 ± 3.08  5 / 10  
FirstConcert              8 / 8  16  64.94 ± 2.93  3 / 13  
HappyDay                 5 / 11  15  65.60 ± 3.09  2 / 13  
HowCouldYou               7 / 9  16  63.75 ± 3.45  4 / 12  
IronyParas                6 / 9  16  64.94 ± 3.62  3 / 13  
Koko                      6 / 9  15  64.60 ± 3.70   6 / 9  
MetaphorsPronouns        5 / 10  16  64.75 ± 3.55  4 / 12  
MetaphorsRepeatedNames   5 / 10  16  65.06 ± 3.45  3 / 13  
ProfessionalDecline       7 / 8  15  64.93 ± 3.41  5 / 10  
Roswell                   8 / 8  15  64.27 ± 3.51  4 / 11  
TaleofTwoCountries       6 / 10  16  66.44 ± 4.11   7 / 9  
TigerandLion              8 / 7  15  65.07 ± 2.76  4 / 11  
UserFriendly              7 / 8  15  63.33 ± 2.79   6 / 9

In [5]:
df_ratings_audio.shape[0]+df_ratings_text.shape[0]

434904

In [124]:
df = df_ratings_text.loc[df_ratings_text['segID'].eq(1)].copy()

# Keep one row per participant
df_sub = df.drop_duplicates(subset='subID')

# def sex_counts(s):
#     x = s.astype(str).str.strip().str.lower()
#     male = x.isin(['male', 'm']).sum()
#     female = x.isin(['female', 'f']).sum()
#     return f"{male}/{female}"   # or f'="{male}/{female}"' for CSV

summary = (
    df_sub.groupby('group')
          .agg(
              N=('subID', 'nunique'),
              mean_age=('age', 'mean'),
              sd_age=('age', 'std'),
              sex=('sex', sex_counts)
          )
)

# Format age
summary['Age'] = summary.apply(lambda r: f"{r['mean_age']:.2f} ± {r['sd_age']:.2f}", axis=1)

# Final columns
summary = summary[['N', 'Age', 'sex']]

summary.columns = ['N', 'Age (mean ± SD)', 'Sex (M/F)']

# Optional ordering
summary = summary.loc[['young', 'middle', 'old']]

summary

,N,Age (mean ± SD),Sex (M/F)
group,,,
young,125,28.04 ± 6.82,48 / 77
middle,114,48.32 ± 5.24,47 / 67
old,73,65.12 ± 3.66,20 / 53


In [126]:
df_sub.groupby(['group','source']).count()

subID  idx  text  text_idx  segID    V    A   RT  age  sex  \
group  source                                                                 
middle Prolific    114  114   114       114    114  114  114  114  114  114   
old    Prolific     73   73    73        73     73   73   73   73   73   73   
young  Prolific     99   99    99        99     99   99   99   99   99   99   
       Sona         26   26    26        26     26   26   26   26   26   26   

                 age_prescreen  sex_prescreen  attention_correct  \
group  source                                                      
middle Prolific            114            114                114   
old    Prolific             73             73                 73   
young  Prolific             99             99                 99   
       Sona                 26             26                 26   

                 attention_correct%  comp_correct  comp_correct%  ethnicity  \
group  source                                                                 
middle Prolific                 114           114            114        114   
old    Prolific                  73            73             73         73   
young  Prolific                  99            99             99         99   
       Sona                      26            26             26         26   

                 residence_states  
group  source                      
middle Prolific               114  
old    Prolific                73  
young  Prolific                99  
       Sona                    26

In [125]:
125+114+73

312